placeholder

---
## 1. Configuração do Ambiente

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import seaborn as sns
import requests
import json
import os
import warnings
from pathlib import Path
from calendar import month_abbr

warnings.filterwarnings('ignore')

# ── Estilo dos gráficos ────────────────────────────────────
sns.set_style('whitegrid')
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'font.family': 'serif',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'legend.fontsize': 9,
})

CORES = {'Sorriso_MT': '#E74C3C', 'Londrina_PR': '#2980B9', 'Rio Verde_GO': '#27AE60'}
CORES_MES = {1: '#08306B', 2: '#08519C', 3: '#2171B5', 4: '#4292C6',
             5: '#6BAED6', 6: '#9ECAE1', 7: '#C6DBEF', 8: '#DEEBF7',
             9: '#FEE090', 10: '#FDCC8A', 11: '#FC8D59', 12: '#E34A33'}

# ── Diretórios ────────────────────────────────────────────
DATA_RAW = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')
OUTPUT = Path('../output/graficos')
for d in [DATA_RAW, DATA_PROCESSED, OUTPUT]:
    d.mkdir(parents=True, exist_ok=True)

# ── Definição das regiões ─────────────────────────────────
REGIOES = {
    'Sorriso_MT':  {'lat': -12.55, 'lon': -55.71, 'estado': 'MT', 'municipio': 'Sorriso'},
    'Londrina_PR': {'lat': -23.31, 'lon': -51.16, 'estado': 'PR', 'municipio': 'Londrina'},
    'Rio Verde_GO': {'lat': -17.80, 'lon': -50.93, 'estado': 'GO', 'municipio': 'Rio Verde'},
}

# NASA POWER — comunidade AG (Agriculture) cobre temperatura e precipitação
PARAMS_NASA = 'T2M,T2M_MAX,T2M_MIN,PRECTOTCORR'
COMMUNITY = 'AG'
START = '19800101'
END = '20241231'

print('✅ Ambiente configurado.')
print(f'Regiões: {list(REGIOES.keys())}')
print(f'Período: {START} a {END}')
print(f'Comunidade: {COMMUNITY} | Parâmetros: {PARAMS_NASA}')

---
## 2. Coleta de Dados — NASA POWER API

A [NASA POWER](https://power.larc.nasa.gov/) fornece séries diárias globais de variáveis meteorológicas em grade de 0.5°×0.5°. Usamos a comunidade `AG` (Agriculture), que cobre tanto temperaturas quanto precipitação.

**Variáveis baixadas:**
- `T2M` — Temperatura média a 2m (°C)
- `T2M_MAX` — Temperatura máxima diária (°C)
- `T2M_MIN` — Temperatura mínima diária (°C)
- `PRECTOTCORR` — Precipitação total diária corrigida (mm)

> ⚠️ A API pública da NASA pode ter limites de taxa. Implementamos caching local para evitar re-download.

In [ ]:
ARQUIVO_CACHE = DATA_RAW / 'nasa_power_raw.parquet'


def baixar_nasa_power(lat, lon, nome_regiao):
    """Baixa dados diários da NASA POWER API e retorna DataFrame."""
    url = (
        f'https://power.larc.nasa.gov/api/temporal/daily/point'
        f'?parameters={PARAMS_NASA}'
        f'&community={COMMUNITY}'
        f'&longitude={lon:.4f}&latitude={lat:.4f}'
        f'&start={START}&end={END}'
        f'&format=JSON'
    )
    try:
        resp = requests.get(url, timeout=90,
                            headers={'Accept': 'application/json'})
        resp.raise_for_status()
        dados = resp.json()['properties']['parameter']
    except Exception as e:
        print(f'  ❌ Falha ao baixar {nome_regiao}: {e}')
        return None

    df = pd.DataFrame({k: pd.Series(v) for k, v in dados.items()})
    df.index = pd.to_datetime(df.index, format='%Y%m%d')
    df.index.name = 'data'
    df.columns = [c.lower() for c in df.columns]
    df['regiao'] = nome_regiao

    print(f'  ✅ {nome_regiao}: {len(df)} dias'
          f'  ({df.index.min().date()} a {df.index.max().date()})')
    return df


# ── Tenta carregar cache local primeiro ───────────────────
if ARQUIVO_CACHE.exists():
    print('📦 Cache local encontrado. Carregando...')
    df_temp = pd.read_parquet(ARQUIVO_CACHE)
    print(f'   {len(df_temp)} registros carregados.')
else:
    print('🌐 Baixando dados da NASA POWER API...')
    dfs = []
    for nome, coords in REGIOES.items():
        print(f'  Região: {nome} ({coords["lat"]:.2f}, {coords["lon"]:.2f})')
        df = baixar_nasa_power(coords['lat'], coords['lon'], nome)
        if df is not None:
            dfs.append(df)

    if not dfs:
        raise RuntimeError('❌ Não foi possível baixar dados de nenhuma região.')

    df_temp = pd.concat(dfs)

    # Salva cache local
    df_temp.to_parquet(ARQUIVO_CACHE)
    print(f'\n📦 Cache salvo em {ARQUIVO_CACHE}')

print(f'\n📊 Dataset final: {len(df_temp)} registros, {df_temp["regiao"].nunique()} regiões')

In [ ]:
# ── Qualidade dos dados ───────────────────────────────────
print('═══ QUALIDADE DOS DADOS ═══')
for regiao in df_temp['regiao'].unique():
    sub = df_temp[df_temp['regiao'] == regiao]
    nulos = sub.isnull().sum()
    total = len(sub)
    print(f'\n{regiao}:')
    print(f'  Registros: {total}')
    print(f'  Período: {sub.index.min().date()} a {sub.index.max().date()}')
    print(f'  Nulos por coluna:')
    for col in ['t2m', 't2m_max', 't2m_min', 'prectotcorr']:
        if col in nulos:
            pct = nulos[col] / total * 100
            print(f'    {col}: {nulos[col]} ({pct:.2f}%)')

# ── Tratamento de missing values ──────────────────────────
cols_temp = ['t2m', 't2m_max', 't2m_min']
for regiao in df_temp['regiao'].unique():
    mask = df_temp['regiao'] == regiao
    # Preenche NAs com interpolação linear (lacunas curtas)
    df_temp.loc[mask, cols_temp] = (
        df_temp.loc[mask, cols_temp].interpolate(method='linear', limit=7)
    )

# Verifica se sobrou algum NA
nas_restantes = df_temp[cols_temp].isnull().sum().sum()
print(f'\nNAs restantes após interpolação: {nas_restantes}')
if nas_restantes > 0:
    df_temp = df_temp.dropna(subset=cols_temp)
    print(f'  Registros removidos: {nas_restantes}')

# ── Variáveis derivadas ───────────────────────────────────
df_temp['t2m_amplitude'] = df_temp['t2m_max'] - df_temp['t2m_min']
df_temp['ano'] = df_temp.index.year
df_temp['mes'] = df_temp.index.month
df_temp['safra'] = np.where(
    df_temp.index.month >= 10,
    df_temp.index.year,
    df_temp.index.year - 1
)

# ── Salva dados processados ───────────────────────────────
df_temp.to_parquet(DATA_PROCESSED / 'temperatura_diaria_MT_PR_GO.parquet')
print(f'\n💾 Dados processados salvos em:'
      f' {DATA_PROCESSED / "temperatura_diaria_MT_PR_GO.parquet"}')

df_temp.head()

---
## 3. Análise Exploratória

Antes de modelar, precisamos entender o comportamento das temperaturas nas três regiões. As perguntas que guiam esta seção:

1. **Sazonalidade:** Qual a amplitude térmica ao longo do ano em cada região?
2. **Tendência:** Há aquecimento visível no período 1980–2025?
3. **Eventos extremos:** Como evoluiu a frequência de dias com temperatura >30°C?
4. **Heterogeneidade espacial:** As regiões se comportam de forma similar?

In [ ]:
# ── Estatísticas descritivas ──────────────────────────────
print('═══ ESTATÍSTICAS DESCRITIVAS — T2M (°C) ═══')
print(df_temp.groupby('regiao')['t2m'].describe().round(2).to_string())

print('\n═══ MÉDIAS MENSAIS (°C) ═══')
tabela_mensal = df_temp.groupby(['regiao', 'mes'])['t2m'].agg(['mean', 'std', 'min', 'max']).round(2)
print(tabela_mensal.to_string())

print('\n═══ DIAS EXTREMOS (>35°C) POR REGIÃO ═══')
for regiao in df_temp['regiao'].unique():
    sub = df_temp[df_temp['regiao'] == regiao]
    total_dias = len(sub)
    dias_extremos = (sub['t2m_max'] > 35).sum()
    print(f'  {regiao}: {dias_extremos} dias ({dias_extremos/total_dias*100:.1f}% do período)')

In [ ]:
FIGURA = '01_serie_temperatura'
print(f'Gerando {FIGURA}...')

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

for ax, (regiao, coords) in zip(axes, REGIOES.items()):
    sub = df_temp[df_temp['regiao'] == regiao].copy()
    cor = CORES[regiao]

    # Série diária (linha fina e transparente)
    ax.plot(sub.index, sub['t2m'],
            color=cor, alpha=0.15, linewidth=0.4, label='Temperatura diária')

    # Média móvel 30 dias (linha espessa)
    sub['t2m_suav'] = sub['t2m'].rolling(30, center=True).mean()
    ax.plot(sub.index, sub['t2m_suav'],
            color=cor, linewidth=1.2, label='Média móvel 30d')

    # Referência: média do período
    media_geral = sub['t2m'].mean()
    ax.axhline(media_geral, color='#333', linestyle='--',
               linewidth=0.7, alpha=0.6, label=f'Média geral: {media_geral:.1f}°C')

    ax.set_ylabel('Temperatura (°C)')
    ax.set_title(f'{coords["municipio"]}/{coords["estado"]}',
                 fontsize=12, fontweight='bold', loc='left')
    ax.legend(loc='upper right', ncol=3, frameon=True, facecolor='white',
              edgecolor='#ddd', framealpha=0.9)
    ax.set_ylim(10, 35)
    ax.yaxis.set_major_locator(mticker.MultipleLocator(5))

axes[-1].set_xlabel('Ano')
axes[-1].xaxis.set_major_locator(mdates.YearLocator(5))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

fig.suptitle('Série Histórica de Temperatura Média Diária (1980–2025)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT / f'{FIGURA}.png', dpi=300)
plt.show()
print(f'  ✅ Salvo: output/graficos/{FIGURA}.png')

In [ ]:
FIGURA = '02_boxplot_mensal'
print(f'Gerando {FIGURA}...')

fig, ax = plt.subplots(figsize=(14, 6))

# Prepara dados
df_plot = df_temp.copy()
df_plot['regiao_label'] = df_plot['regiao'].map(
    lambda r: f'{REGIOES[r]["municipio"]}/{REGIOES[r]["estado"]}'
)

# Ordem das regiões (latitude decrescente → norte para sul)
ordem_regioes = ['Sorriso_MT', 'Rio Verde_GO', 'Londrina_PR']
ordem_labels = [f'{REGIOES[r]["municipio"]}/{REGIOES[r]["estado"]}'
               for r in ordem_regioes]

sns.boxplot(
    data=df_plot,
    x='mes', y='t2m', hue='regiao',
    hue_order=ordem_regioes,
    palette=CORES,
    ax=ax, width=0.7, linewidth=0.6, fliersize=1.5
)

# Linhas de referência térmica da soja
ax.axhline(30, color='#E74C3C', linestyle=':', linewidth=0.8, alpha=0.7,
           label='Limiar crítico soja (30°C)')
ax.axhline(25, color='#27AE60', linestyle=':', linewidth=0.8, alpha=0.7,
           label='Temperatura ótima (25°C)')

ax.set_xlabel('Mês')
ax.set_ylabel('Temperatura média (°C)')
ax.set_title('Distribuição Mensal da Temperatura por Região',
             fontsize=13, fontweight='bold', loc='left')
ax.set_xticklabels([month_abbr[i] for i in range(1, 13)])
ax.legend(loc='upper right', frameon=True, facecolor='white',
          edgecolor='#ddd', framealpha=0.9)

plt.tight_layout()
plt.savefig(OUTPUT / f'{FIGURA}.png', dpi=300)
plt.show()
print(f'  ✅ Salvo: output/graficos/{FIGURA}.png')

In [ ]:
FIGURA = '03_anomalias_heatmap'
print(f'Gerando {FIGURA}...')

# Calcula anomalia: temperatura média mensal - climatologia (1981-2010)
CLIMATOLOGIA_INI = 1981
CLIMATOLOGIA_FIM = 2010

# Filtra período climatológico
mask_clim = ((df_temp.index.year >= CLIMATOLOGIA_INI) &
             (df_temp.index.year <= CLIMATOLOGIA_FIM))

# Normal climatológica (média por mês para cada região)
climatologia = (
    df_temp[mask_clim]
    .groupby(['regiao', 'mes'])['t2m']
    .mean()
    .rename('clim_norm')
)

# Temperatura média mensal por região/ano/mês
media_mensal = (
    df_temp
    .groupby(['regiao', 'ano', 'mes'])['t2m']
    .mean()
    .rename('temp_mensal')
    .reset_index()
)

# Junta com climatologia
media_mensal = media_mensal.merge(
    climatologia.reset_index(), on=['regiao', 'mes'], how='left'
)
media_mensal['anomalia'] = media_mensal['temp_mensal'] - media_mensal['clim_norm']

# Heatmap para cada região
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

for ax, (regiao, coords) in zip(axes, REGIOES.items()):
    sub = media_mensal[media_mensal['regiao'] == regiao].pivot_table(
        index='ano', columns='mes', values='anomalia'
    )
    sub = sub.dropna(how='all', axis=0)

    vmax = max(abs(sub.min().min()), abs(sub.max().max()))

    sns.heatmap(sub, ax=ax, cmap='RdBu_r', center=0,
                vmin=-vmax, vmax=vmax,
                cbar_kws={'label': 'Anomalia (°C)', 'shrink': 0.6},
                linewidths=0, xticklabels=list(month_abbr[1:]))

    ax.set_title(f'{coords["municipio"]}/{coords["estado"]}',
                 fontsize=12, fontweight='bold', loc='left')
    ax.set_ylabel('Ano')
    ax.set_xlabel('')

fig.suptitle('Anomalias Térmicas Mensais em Relação à Climatologia 1981–2010',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT / f'{FIGURA}.png', dpi=300)
plt.show()
print(f'  ✅ Salvo: output/graficos/{FIGURA}.png')

In [ ]:
FIGURA = '04_dias_acima_30c_safra'
print(f'Gerando {FIGURA}...')

# Conta dias com T_max > 30°C no período crítico da soja (DEZ-FEV)
# Período crítico = floração e enchimento de grãos
mask_critico = df_temp['mes'].isin([12, 1, 2])
df_critico = df_temp[mask_critico].copy()

# Ajusta safra: dezembro pertence à safra do ano seguinte
df_critico['safra_ajustada'] = np.where(
    df_critico['mes'] == 12,
    df_critico['ano'] + 1,
    df_critico['ano']
)

# Contagem de dias >30°C por safra e região
dias_extremos = (
    df_critico[df_critico['t2m_max'] > 30]
    .groupby(['regiao', 'safra_ajustada'])
    .size()
    .rename('dias_acima_30')
    .reset_index()
    .dropna()
)

# Preenche safras sem dias extremos com 0
todas_safras = range(
    dias_extremos['safra_ajustada'].min().astype(int),
    dias_extremos['safra_ajustada'].max().astype(int) + 1
)
full_index = pd.MultiIndex.from_product(
    [list(REGIOES.keys()), todas_safras],
    names=['regiao', 'safra_ajustada']
)
dias_extremos = (
    dias_extremos.set_index(['regiao', 'safra_ajustada'])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

# Gráfico
fig, ax = plt.subplots(figsize=(14, 6))

for regiao in ordem_regioes:
    sub = dias_extremos[dias_extremos['regiao'] == regiao]
    cor = CORES[regiao]
    label = f'{REGIOES[regiao]["municipio"]}/{REGIOES[regiao]["estado"]}'

    ax.bar(sub['safra_ajustada'] - 0.25 + 0.25 * ordem_regioes.index(regiao),
           sub['dias_acima_30'],
           width=0.2, color=cor, alpha=0.8, label=label, edgecolor='white', linewidth=0.3)

    # Tendência linear
    from scipy.stats import linregress
    x = sub['safra_ajustada'].astype(int).values
    y = sub['dias_acima_30'].values
    slope, intercept, r_val, p_val, _ = linregress(x, y)
    x_trend = np.array([x.min(), x.max()])
    y_trend = intercept + slope * x_trend
    ax.plot(x_trend, y_trend, color=cor, linewidth=1.8, linestyle='--', alpha=0.7)
    print(f'  {label}: tendência de {slope:.2f} dias/ano (p={p_val:.3f})')

ax.set_xlabel('Safra')
ax.set_ylabel('Dias com Tmax > 30°C (DEZ-FEV)')
ax.set_title('Evolução de Dias Extremamente Quentes no Período Crítico da Safra (DEZ-FEV)',
             fontsize=12, fontweight='bold', loc='left')
ax.legend(loc='upper left', frameon=True, facecolor='white',
          edgecolor='#ddd', framealpha=0.9)

plt.tight_layout()
plt.savefig(OUTPUT / f'{FIGURA}.png', dpi=300)
plt.show()
print(f'  ✅ Salvo: output/graficos/{FIGURA}.png')

---
## 4. Síntese dos Resultados

### Principais observações:

1. **Sazonalidade:** As três regiões apresentam padrão sazonal claro, com temperaturas mais altas entre setembro e março (primavera/verão) e mais amenas entre maio e agosto. Sorriso/MT mantém-se consistentemente acima de 25°C mesmo nos meses mais frios.

2. **Gradiente latitudinal:** A temperatura média decresce com o aumento da latitude (não capturado diretamente, mas visível na comparação MT > GO > PR).

3. **Eventos extremos:** A frequência de dias com Tmax > 30°C no período crítico da soja (DEZ-FEV) mostra tendência de alta nas três regiões, corroborando a literatura sobre aquecimento no Centro-Oeste brasileiro.

4. **Anomalias:** O heatmap de anomalias térmicas permite identificar visualmente anos com estresse térmico atípico (ex.: El Niño/La Niña).

### Próximos passos (Notebook 2):
- Modelar a temperatura como processo estocástico (Ornstein-Uhlenbeck)
- Calibrar parâmetros por máxima verossimilhança
- Validar a capacidade do modelo de reproduzir as estatísticas observadas